здесь сильные модели проверяют ответы тестируемых моделей на правильность.
реализовано сравнение эталонного ответа из бенчмарка и ответа модели, потому что метрики типа f1 не всегда помогают в задачах типа qa и summarization. 


llm as a judge

In [1]:
import pandas as pd
from tqdm import tqdm
import json

загрузка датасета

In [2]:
dataset_path = "/kaggle/input/datasets/yumiasakura/test-800/test_results (1).csv"
df = pd.read_csv(dataset_path)

print(df.head())

print(df.info())

    task_id benchmark task_type dataset_split  \
0  bbh_1263       bbh        qa          test   
1  bbh_3951       bbh        qa          test   
2  bbh_6199       bbh        qa          test   
3  bbh_6080       bbh        qa          test   
4    bbh_96       bbh        qa          test   

                                            question context options  \
0  "It is not always easy to grasp who is consumi...     NaN     NaN   
1  On the table, you see two burgundy mugs, one b...     NaN     NaN   
2  Question: Rashida tells the truth. Jim says Ra...     NaN     NaN   
3  Question: Phoebe tells the truth. Jamey says P...     NaN     NaN   
4                  False or ( False ) or not True is     NaN     NaN   

  correct_answer subtask                                          metadata  \
0          valid     NaN                 {'task_name': 'formal_fallacies'}   
1            (C)     NaN  {'task_name': 'reasoning_about_colored_objects'}   
2             No     NaN              

In [31]:
#df['task_type'] = df['task_type'].replace('multiple_choice', 'qa')

In [3]:
df.head()

,task_id,benchmark,task_type,dataset_split,question,context,options,correct_answer,subtask,metadata,Qwen-7b_1,Qwen-7b_1_result,Qwen-7b_2,Qwen-7b_2_result,Qwen-7b_3,Qwen-7b_3_result
0,bbh_1263,bbh,qa,test,"""It is not always easy to grasp who is consumi...",NaN,NaN,valid,NaN,{'task_name': 'formal_fallacies'},Answer:\n\nThe argument is invalid. The premis...,NaN,Answer:\n\nThe argument is invalid. The premis...,NaN,Answer:\n\nThe argument is invalid. The premis...,NaN
1,bbh_3951,bbh,qa,test,"On the table, you see two burgundy mugs, one b...",NaN,NaN,(C),NaN,{'task_name': 'reasoning_about_colored_objects'},(R) seventeen\n(S) eighteen\n(T) nineteen\n(U)...,NaN,(R) seventeen\n(S) eighteen\n(T) nineteen\n(U)...,NaN,(R) seventeen\n(S) eighteen\n(T) nineteen\n(U)...,NaN
2,bbh_6199,bbh,qa,test,Question: Rashida tells the truth. Jim says Ra...,NaN,NaN,No,NaN,{'task_name': 'web_of_lies'},A:\n\nVina tells the truth.,NaN,A:\n\nVina tells the truth.,NaN,A:\n\nVina tells the truth.,NaN
3,bbh_6080,bbh,qa,test,Question: Phoebe tells the truth. Jamey says P...,NaN,NaN,No,NaN,{'task_name': 'web_of_lies'},A:\n\nVina tells the truth.,NaN,A:\n\nVina tells the truth.,NaN,A:\n\nVina tells the truth.,NaN
4,bbh_96,bbh,qa,test,False or ( False ) or not True is,NaN,NaN,False,NaN,{'task_name': 'boolean_expressions'},Answer:\nFalse,NaN,Answer:\nFalse,NaN,Answer:\nFalse,NaN


оценивание результатов

In [47]:
def evaluate_answer(row, pred):

    task = row["task_type"]
    gold = row["correct_answer"]
    benchmark = row["benchmark"]
    question = row["question"]
    context = row["context"]

    if task == "multiple_choice" or task == "fact_verification":
        if benchmark == "hellaswag" or benchmark == "hover" or benchmark == "fever" or benchmark == "mmlu":
            if benchmark == "mmlu":
                pred = pred[7:10]
            return check_correct_mc(row, pred)
        else:

            pred_option = extract_choice(pred)
            return pred_option == gold

    elif task == "qa":
        #if extract_match(pred, gold):
            #return True
        if benchmark == "halueval" or benchmark == "truthfulqa" or benchmark == "bbh":
            f1_metric = f1_score(pred, gold)
            #print(f1_metric)
            if check_correct_mc(row, pred):
                return True
            elif f1_metric>=0.7:
                return True
            else: 
                verdict_llm = llm_judge_qa(question, gold, pred, judge_model, judge_tokenizer)
                if verdict_llm == True:
                    return True
                else:
                    return False
                	
    elif task == "summarization":
        cosine_similarity = semantic_match(pred, gold)
        rouge_scores = compute_rouge(pred, gold)
        r1 = rouge_scores["rouge1"]
        r2 = rouge_scores["rouge2"]
        rL = rouge_scores["rougeL"]
        #print(cosine_similarity)
        if rL>=0.35 or (r1>=0.45 and r2>=0.2):
            return True
        elif (rL>=0.2) or (r1>=0.3):
            verdict_llm = llm_judge_summary(question, context, gold, pred, judge_model, judge_tokenizer)
            if verdict_llm  == True:
                return True
        elif cosine_similarity>=0.7:
            return True
        elif cosine_similarity>=0.3 and cosine_similarity<0.7:
            verdict_llm = llm_judge_summary(question, context, gold, pred, judge_model, judge_tokenizer)
            if verdict_llm  == True:
                return True
            else:
                return False
        else: 
            return False


    return False

In [5]:
# Проверка корректности ответа для задачи mc

def check_correct_mc(row, pred):

    if row["correct_answer"] is None:
        return None

    gt = str(row["correct_answer"]).lower()
    pred = pred.lower()
    #print(gt in pred)

    return gt in pred


In [53]:
df["Qwen-7b_1"].isna().sum()

np.int64(4)

In [54]:
pd.concat([df.iloc[170:185]])

,task_id,benchmark,task_type,dataset_split,question,context,options,correct_answer,subtask,metadata,Qwen-7b_1,Qwen-7b_1_result,Qwen-7b_2,Qwen-7b_2_result,Qwen-7b_3,Qwen-7b_3_result
170,dailymail_8318,dailymail,summarization,validation,NaN,Evidence: This is a picture of the stolen Budd...,NaN,Villagers say relic is their 'ancestral patria...,NaN,NaN,Summary:\n\nChinese villagers are demanding th...,False,Summary:\n\nChinese villagers are demanding th...,NaN,Summary:\n\nChinese villagers are demanding th...,NaN
171,dailymail_6385,dailymail,summarization,validation,NaN,Marine biologist: Diane Cowan has worked with ...,NaN,"Diane Cowan, 54, has been stuck on Friendship ...",NaN,NaN,Summary:\n\nMarine biologist Diane Cowan has b...,True,Summary:\n\nMarine biologist Diane Cowan has b...,NaN,Summary:\n\nMarine biologist Diane Cowan has b...,NaN
172,dailymail_4209,dailymail,summarization,validation,NaN,When England crashed out of the last global li...,NaN,England face Afghanistan with more to lose tha...,NaN,NaN,The England cricket team is facing a final mat...,False,The England cricket team is facing a final mat...,NaN,The England cricket team is facing a final mat...,NaN
173,dailymail_10159,dailymail,summarization,validation,NaN,For most of her life young Queensland woman Sa...,NaN,"US father never gave up on daughter,Savanna To...",NaN,NaN,"The article is about Samantha Geldenhuys, who ...",False,"The article is about Samantha Geldenhuys, who ...",NaN,"The article is about Samantha Geldenhuys, who ...",NaN
174,dailymail_8949,dailymail,summarization,validation,NaN,Remembered in history by some as a murderous t...,NaN,Mr Cumberbatch and Richard III are second cous...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
175,dailymail_5687,dailymail,summarization,validation,NaN,A Chinese family was kicked off a Cathay Pacif...,NaN,Plane sat at its gate in Bangkok while father ...,NaN,NaN,The article reports on a Chinese family who we...,NaN,The article reports on a Chinese family who we...,NaN,The article reports on a Chinese family who we...,NaN
176,dailymail_6613,dailymail,summarization,validation,NaN,A watchdog has cleared officers over the case ...,NaN,Hamzah Khan's decomposed body found in cot two...,NaN,NaN,Summary:\n\nA watchdog has cleared West Yorksh...,NaN,Summary:\n\nA watchdog has cleared West Yorksh...,NaN,Summary:\n\nA watchdog has cleared West Yorksh...,NaN
177,dailymail_63,dailymail,summarization,validation,NaN,"(CNN)The Rev. Fred Craddock, the pulpit giant ...",NaN,Fred Craddock revolutionized art of preaching ...,NaN,NaN,"The Rev. Fred Craddock, a renowned preacher, h...",NaN,"The Rev. Fred Craddock, a renowned preacher, h...",NaN,"The Rev. Fred Craddock, a renowned preacher, h...",NaN
178,dailymail_10193,dailymail,summarization,validation,NaN,Enjoy a dip in the sea and you risk picking up...,NaN,Swimmers and surfers are most at risk of catch...,NaN,NaN,Summary:\n\nA study by the University of Exete...,NaN,Summary:\n\nA study by the University of Exete...,NaN,Summary:\n\nA study by the University of Exete...,NaN
179,dailymail_6423,dailymail,summarization,validation,NaN,A 23-year-old woman allegedly caught driving w...,NaN,Bianca Micallef was stopped by police on Tuesd...,NaN,NaN,"A 23-year-old woman, Bianca Micallef, has been...",NaN,"A 23-year-old woman, Bianca Micallef, has been...",NaN,"A 23-year-old woman, Bianca Micallef, has been...",NaN


In [55]:
df["Qwen-7b_1"] = df["Qwen-7b_1"].fillna("")

In [56]:
df["Qwen-7b_1"].isna().sum()

np.int64(0)

In [57]:
df["Qwen-7b_2"].isna().sum()

np.int64(4)

In [58]:
df["Qwen-7b_3"].isna().sum()

np.int64(4)

In [59]:
df["Qwen-7b_2"] = df["Qwen-7b_3"].fillna("")
df["Qwen-7b_3"] = df["Qwen-7b_3"].fillna("")

In [6]:
from collections import Counter
import re
import string

In [7]:
def extract_choice(text):
    match = re.search(r'\b([A-D])\b', text)
    if match:
        return match.group(1)
    match = re.search(r'\b([0-3])\b', text)
    if match:
        return match.group(1)
    return None

In [32]:
s = 5
if s is float or s is int:
    print(7)

In [43]:
def is_number(s):
    try:
        float(s)
        return True
    except ValueError:
        return False

In [44]:
def normalize(text):
    text = str(text)
    if is_number(text) == False:
        text = text.lower()
    text = re.sub(r'\b(a|an|the)\b', ' ', text)
    text = text.translate(str.maketrans('', '', string.punctuation))
    text = " ".join(text.split())

    return text

In [42]:
normalize(8)

'8'

In [9]:
# Проверка корректности ответа для задачи qa

def f1_score(pred, gold):

    pred_tokens = normalize(pred).split()
    gold_tokens = normalize(gold).split()
    #print(pred_tokens)
    #print(gold_tokens)
    if len(pred_tokens)>0:
        if pred_tokens[0]=="answer":
            pred_tokens = pred_tokens[1:]

    common = Counter(pred_tokens) & Counter(gold_tokens)
    num_same = sum(common.values())

    if num_same == 0:
        return 0

    precision = num_same / len(pred_tokens)
    recall = num_same / len(gold_tokens)
    
    return 2 * precision * recall / (precision + recall)



llm as a judge 

Llama 3-7b - хорошо следует инструкциям из промпта, дает стабильные оценки

In [10]:
def build_judge_prompt_qa(question, correct_answer, model_answer):

    prompt = f"""
You are a strict evaluator of question answering systems.

Your task is to determine whether the model answer is correct.

Question:
{question}

Ground truth answer:
{correct_answer}

Model answer:
{model_answer}

Rules:
- Answer YES only if the meaning is clearly the same.
- Minor wording differences are allowed.
- Answer NO if there is any contradiction, missing key info, or wrong facts.
- Be strict.

Reply with only one word: YES or NO.
Do not explain.
"""

    return prompt

In [11]:
def generate_judge_answer(prompt, judge_model, judge_tokenizer):

    messages = [
        {"role": "system", "content": "You are a strict and precise judge."},
        {"role": "user", "content": prompt}
    ]

    input_ids = judge_tokenizer.apply_chat_template(
        messages,
        return_tensors="pt"
    ).to(judge_model.device)

    with torch.no_grad():
        output = judge_model.generate(
        **input_ids,   
        max_new_tokens=30,
        temperature=0.0,
        do_sample=False,
        pad_token_id=judge_tokenizer.eos_token_id
        )

    decoded = judge_tokenizer.decode(output[0], skip_special_tokens=True)
    #print(decoded)

    # берём только сгенерированную часть
    response = decoded.split("assistant")[-1].strip()

    return response

In [12]:
def llm_judge_qa(question, correct_answer, model_answer, judge_model, judge_tokenizer):

    prompt = build_judge_prompt_qa(
        question,
        correct_answer,
        model_answer
    )
    #print("PROMPT ", prompt)

    raw_output = generate_judge_answer(
        prompt,
        judge_model,
        judge_tokenizer
    )
    #print("RAW_OUTPUT ", raw_output)

    result = parse_judge_output(raw_output)
    
    #print("RESULT ", result)

    return result, raw_output

In [13]:
def llm_judge_summary(question, context, correct_answer, model_answer, judge_model, judge_tokenizer):
                     
    prompt = build_judge_prompt_sum(
        question,
        context,
        correct_answer,
        model_answer
    )

    raw = generate_judge_answer(
        prompt, 
        judge_model,
        judge_tokenizer
    )
    
    #print(raw)
    return parse_judge_output(raw), raw

In [14]:
def build_judge_prompt_sum(question, context, correct_answer, model_answer):

    prompt = f"""
You are an expert evaluator of summarization systems.

Evaluate the model summary based on the document and the reference summary.

Question:
{question}

Document:
{context}

Reference summary:
{correct_answer}

Model summary:
{model_answer}

Evaluate the model summary on the following criteria:

1. Faithfulness (no hallucinations, all facts supported by the document)
2. Correctness (facts match the reference summary)
3. Coverage (important information is included)
4. Conciseness (no unnecessary details)

Give your final verdict:

- YES → if the summary is overall correct and faithful
- NO → if it contains hallucinations or major errors

Be strict.

Reply with only one word: YES or NO.
Do not explain.
"""
    return prompt

In [15]:
def parse_judge_output(text):

    text = text.strip().upper()

    if text.startswith("YES"):
        return True
    if text.startswith("NO"):
        return False

    return None

метрика семантическая схожесть

In [16]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

In [17]:
model_SentenceTransformer = SentenceTransformer("all-MiniLM-L6-v2")   

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [18]:
def semantic_match(pred, gold):

    emb = model_SentenceTransformer.encode([pred, gold])

    sim = cosine_similarity([emb[0]], [emb[1]])[0][0]

    return sim       

загрузка модели для подхода llm as a judge

In [2]:
!pip install -U bitsandbytes>=0.46.1

In [19]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

In [20]:
from huggingface_hub import login

login("hf_LOGIN")

In [21]:
global judge_tokenizer
global judge_model

In [22]:
model_name = "meta-llama/Meta-Llama-3-8B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

judge_tokenizer = AutoTokenizer.from_pretrained(model_name, token=True)
judge_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    token=True
)
judge_tokenizer.pad_token = judge_tokenizer.eos_token

config.json:   0%|          | 0.00/654 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/73.0 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/187 [00:00<?, ?B/s]

метрика rouge для summarization

In [23]:
from rouge_score import rouge_scorer

In [24]:
scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'],use_stemmer=True)

def compute_rouge(pred, gold):
    scores = scorer.score(gold, pred)

    return {
        "rouge1": scores["rouge1"].fmeasure,
        "rouge2": scores["rouge2"].fmeasure,
        "rougeL": scores["rougeL"].fmeasure,
    }

метрика BERTscore для суммаризации и длинных ответов qa

In [41]:
!pip install bert_score

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 2.0 MB/s eta 0:00:00


In [45]:
from bert_score import score as bertscore

In [46]:
def compute_bertscore(pred, gold):
    P, R, F1 = bertscore(
        [pred],
        [gold],
        lang="en",   # важно для CNN/DailyMail и большинства QA
        rescale_with_baseline=True
    )
    return float(F1[0])

оценка ответов

In [25]:
attempt = 3

In [62]:
results = []
k = 0
for i in range(1, attempt+1):
    answer_col = f"Qwen-7b_{i}"
    result_col = f"Qwen-7b_{i}_result"
    
    for idx, row in tqdm(df.iterrows(), total=len(df)):
        
        answer = row[answer_col]
        #k+=1
        #if k>170:
            #print(answer)
        is_correct = evaluate_answer(row, answer)
        #print("IS CORRECT", is_correct)
       
        df.at[idx, result_col] = is_correct

  0%|          | 0/800 [00:00<?, ?it/s]/tmp/ipykernel_118/1930262150.py:16: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'True' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.at[idx, result_col] = is_correct
  0%|          | 0/800 [00:00<?, ?it/s]/tmp/ipykernel_118/1930262150.py:16: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'True' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.at[idx, result_col] = is_correct
100%|██████████| 800/800 [07:03<00:00,  1.89it/s]


In [30]:
print(df[["correct_answer", "Qwen-7b_1"]])

                                       correct_answer  \
0   Zully Broussard decided to give a kidney to a ...   
1   The 20th MLS season begins this weekend .\nLea...   
2   Bafetimbi Gomis collapses within 10 minutes of...   
3   Rory McIlroy throws club into water at WGC Cad...   
4   Cayman Naib, 13, hasn't been heard from since ...   
5   Ruben Navarrette: Schilling deserves praise fo...   
6   Two American women arrested for carving initia...   
7   It will be a first time for the tour stateside...   
8   A jihadist group claims responsibility in an a...   
9   Alleged incident happened in match at St James...   
10  This page includes the show Transcript .\nUse ...   
11  The abducted workers were seized in an attack ...   
12  Wichita, Kansas, high school students surprise...   
13  Comic book artist Norman Lee went missing in t...   
14  This page includes the show Transcript .\nUse ...   
15  Jeff Goldblum set to reprise his role in "Inde...   
16  Walter Mondale was released

In [31]:
pred = df["correct_answer"][1]
gold = df["Qwen-7b_1"][1]
print(pred)
print(gold)

The 20th MLS season begins this weekend .
League has changed dramatically since its inception in 1996 .
Some question whether rules regarding salary caps and transfers need to change .
The MLS has progressed significantly since its inception in 1996, with higher attendances, more teams, and a new domestic TV and media rights deal worth $700 million over eight years. The league has also attracted more prominent stars


In [32]:
semantic_match(pred, gold)

np.float32(0.5730698)

In [33]:
f1_score(pred, gold)

0.2903225806451613

In [34]:
compute_rouge(pred, gold)

{'rouge1': 0.30303030303030304,
 'rouge2': 0.15625,
 'rougeL': 0.2727272727272727}

In [47]:
compute_bertscore(pred, gold)

config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.bias              | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


0.24950893223285675

accuracy по бенчмаркам

In [63]:
for i in range(1, attempt+1):
    result_col = f"Qwen-7b_{i}_result"
    accuracy = df.groupby("benchmark")[result_col].mean()

    print(accuracy)

benchmark
bbh           0.43
dailymail     0.34
fever          0.5
halueval       0.3
hellaswag     0.18
hover         0.52
mmlu          0.29
truthfulqa    0.16
Name: Qwen-7b_1_result, dtype: object
benchmark
bbh           0.43
dailymail     0.34
fever          0.5
halueval       0.3
hellaswag     0.18
hover         0.52
mmlu          0.29
truthfulqa    0.16
Name: Qwen-7b_2_result, dtype: object
benchmark
bbh           0.43
dailymail     0.34
fever          0.5
halueval       0.3
hellaswag     0.18
hover         0.52
mmlu          0.29
truthfulqa    0.16
Name: Qwen-7b_3_result, dtype: object


In [58]:
df.to_csv(
    "/kaggle/working/truthfulqa_hellaswag_qwen_results.csv",
    index=False
)